# Time Series Analysis by Primary Diagnosis (AutoML)

Time series by primary diagnosis: selection of viable series by temporal density, **AutoML** (FLAML) modeling to choose the best model per series, and export of summaries, charts, and evaluation metrics.

## 1. Import data

**Summary:** Loads the classified Excel, filters to the period 2022-01-01 to 2022-12-31 (52 weeks), and aggregates counts by **week** and **primary_diagnosis** (disease). Lacunas (semanas sem casos) são preenchidas com média móvel. Output is a weekly table (week, doenca, casos) used in the next steps.

In [1]:
import os
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
from pathlib import Path

ROOT = Path.cwd().resolve()
for _ in range(6):
    if (ROOT / "data").exists() and (ROOT / "notebooks").exists():
        break
    ROOT = ROOT.parent if ROOT.parent != ROOT else ROOT

PATH_INPUT = ROOT / "data" / "results" / "agent_outputs" / "agent_outputs_dataset_sintomas_grupos_classificado.xlsx"
PATH_OUTPUT = ROOT / "data" / "results" / "time_series_outputs"
PATH_SUMMARIES = PATH_OUTPUT / "summaries"
PATH_CHARTS = PATH_OUTPUT / "charts"
os.makedirs(PATH_OUTPUT, exist_ok=True)
os.makedirs(PATH_SUMMARIES, exist_ok=True)
os.makedirs(PATH_CHARTS, exist_ok=True)

DATE_START = "2022-01-01"
DATE_END = "2022-12-31"  # 52 semanas no ano
FREQ = "W-MON"
N_WEEKS = 52
TEST_START_WEEK = 37   # treino = semanas 1-36, teste = semanas 37-52

df = pd.read_excel(PATH_INPUT, engine="openpyxl")
df["report_created_at"] = pd.to_datetime(df["report_created_at"], errors="coerce")
df = df.dropna(subset=["report_created_at", "primary_diagnosis"])
df = df[(df["report_created_at"] >= DATE_START) & (df["report_created_at"] <= DATE_END)]
df["week"] = df["report_created_at"].dt.to_period(FREQ).dt.start_time

if "user_id" in df.columns:
    weekly = df.groupby(["week", "primary_diagnosis"])["user_id"].nunique().reset_index()
else:
    weekly = df.groupby(["week", "primary_diagnosis"]).size().reset_index(name="n")
    weekly["user_id"] = weekly["n"]
weekly.columns = ["week", "doenca", "casos"]

print(f"Registros semanais: {len(weekly)}")
print(f"Doenças: {weekly["doenca"].nunique()}")
weekly.head(10)

Registros semanais: 1320
Doenças: 86


,week,doenca,casos
0,2021-12-28,Abscesso da faringe,1
1,2021-12-28,Conjuntivite bacteriana,1
2,2021-12-28,Deficiência de glicocorticoide,1
3,2021-12-28,Deficiência enzimática da glicose-6-fosfato de...,3
4,2021-12-28,Desatelação pulmonar,1
5,2021-12-28,Distrofia muscular,4
6,2021-12-28,Doença das glândulas salivares,1
7,2021-12-28,Doença do tecido conjuntivo,1
8,2021-12-28,Fibrose cística,2
9,2021-12-28,Gripe,1


## 2. Select series with viable density

**Summary:** For each disease, computes **temporal density** (share of weeks with at least one case) and total cases. Keeps only series with **at least 30 weeks with data** and minimum total cases (>= 20) so that time series modeling is feasible.

In [2]:
weeks_full = pd.date_range(start=DATE_START, end=DATE_END, freq=FREQ)
n_weeks = len(weeks_full)

density_list = []
for doenca, grp in weekly.groupby("doenca"):
    n_weeks_with_data = grp["week"].nunique()
    total_casos = grp["casos"].sum()
    density = n_weeks_with_data / n_weeks if n_weeks else 0
    density_list.append({"doenca": doenca, "density": density, "total_casos": total_casos, "n_weeks_with_data": n_weeks_with_data})

density_df = pd.DataFrame(density_list)
MIN_SEMANAS = 30  # pelo menos 30 semanas com dados para modelar
MIN_CASOS = 20
viable = density_df[(density_df["n_weeks_with_data"] >= MIN_SEMANAS) & (density_df["total_casos"] >= MIN_CASOS)]["doenca"].tolist()
print(f"Séries viáveis (>= {MIN_SEMANAS} semanas com dados, total_casos >= {MIN_CASOS}): {len(viable)}")
density_df.sort_values("density", ascending=False).head(15)

Séries viáveis (>= 30 semanas com dados, total_casos >= 20): 16


,doenca,density,total_casos,n_weeks_with_data
56,Micotemia infecciosa,0.846154,985,44
20,Distrofia muscular,0.788462,518,41
48,Hipertensão arterial,0.769231,268,40
73,Tricomonose,0.730769,456,38
23,Doença das glândulas salivares,0.711538,348,37
72,Tosse convulsa,0.692308,371,36
81,piolho,0.692308,365,36
45,Hipergamalglobulinemia,0.692308,241,36
17,Deficiência enzimática da glicose-6-fosfato de...,0.692308,417,36
32,Febre do dengue,0.634615,477,33


## 3. AutoML modeling (70% train / 30% test)

**Summary:** Séries viáveis têm pelo menos 30 semanas com dados. As **lacunas** (semanas sem casos) são preenchidas com **média móvel** (janela 3, centralizada). Cada série tem 52 semanas: **treino = semanas 1–36**, **teste = semanas 37–52**. **FLAML** (ts_forecast) escolhe o melhor modelo; geração de forecast e métricas (RMSE, MAE, MAPE). Séries de má qualidade (ex.: MAPE alto ou IC95% cruzando zero) são sinalizadas.

In [3]:
from flaml import AutoML
from sklearn.metrics import mean_squared_error, mean_absolute_error
import matplotlib.pyplot as plt

def safe_name(s, max_len=50):
    return s.replace("/", "-").replace("\\", "-")[:max_len]

def rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))

def mape(y_true, y_pred):
    mask = y_true != 0
    if not mask.any():
        return np.nan
    return np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100

automl_results = []
TIME_BUDGET = 90

for i, doenca in enumerate(viable):
    print(f"  [{i+1}/{len(viable)}] {doenca}")
    sub = weekly[weekly["doenca"] == doenca].set_index("week")["casos"]
    series = sub.reindex(pd.DatetimeIndex(weeks_full)).fillna(0)
    # Preencher lacunas (semanas sem dados) com média móvel (janela 3, centralizada)
    mask_gap = (series == 0)
    if mask_gap.any():
        rol = series.rolling(3, min_periods=1, center=True).mean()
        series = series.copy()
        series[mask_gap] = rol[mask_gap]
        # Bordas que ainda ficaram zero: preencher com ffill/bfill
        if series.eq(0).any():
            series = series.replace(0, np.nan).ffill().bfill().fillna(0)
    series = series.astype(float)
    n = len(series)
    if n < N_WEEKS:
        continue
    cut = TEST_START_WEEK - 1  # treino = semanas 1 a 36 (índices 0..35)
    train_series = series.iloc[:cut]
    test_series = series.iloc[cut:]
    train_df = train_series.reset_index()
    train_df.columns = ["timestamp", "value"]
    period = len(test_series)  # 16 semanas (37 a 52)
    test_series_used = test_series

    try:
        automl = AutoML()
        automl.fit(
            dataframe=train_df,
            label="value",
            period=period,
            task="ts_forecast",
            time_budget=TIME_BUDGET,
            metric="mape",
            eval_method="holdout",
            verbose=0,
        )
        X_test = test_series_used.index.to_frame(index=False)
        X_test.columns = ["timestamp"]
        y_pred = automl.predict(X_test)
        y_pred = np.maximum(np.asarray(y_pred).flatten(), 0)
        y_test = test_series_used.values
    except Exception as e:
        print(f"    Erro: {e}")
        continue

    res_std = np.nanstd(y_test - y_pred)
    if np.isnan(res_std) or res_std <= 0:
        res_std = np.nanstd(y_test) or 1.0
    lower = np.maximum(y_pred - 1.96 * res_std, 0)
    upper = y_pred + 1.96 * res_std

    best_estimator = getattr(automl, "model", None)
    try:
        cfg = getattr(automl, "best_config", None)
        model_name = cfg.get("learner") if isinstance(cfg, dict) else None
    except Exception:
        model_name = None
    if not model_name:
        est = getattr(best_estimator, "estimator", best_estimator)
        model_name = type(est).__name__ if est is not None else "AutoML"
    model_name = str(model_name)

    summary_text = ""
    try:
        est = getattr(best_estimator, "estimator", best_estimator)
        if hasattr(est, "summary"):
            s = est.summary()
            summary_text = s.as_text() if hasattr(s, "as_text") else str(s)
        else:
            summary_text = f"Model: {model_name}"
    except Exception:
        summary_text = f"Model: {model_name}"

    rmse_val = rmse(y_test, y_pred)
    mape_val = mape(y_test, y_pred)
    mae_val = mean_absolute_error(y_test, y_pred)
    ci_crosses_zero = np.any((y_pred - 1.96 * res_std) <= 0)
    bad_metrics = (np.isfinite(mape_val) and mape_val > 100) or (np.isfinite(rmse_val) and rmse_val > 2 * np.nanmean(y_test))
    if ci_crosses_zero or bad_metrics:
        obs = "Série muito ruim: "
        parts = []
        if bad_metrics:
            parts.append("métricas de desempenho elevadas (MAPE>100% ou RMSE muito alto)")
        if ci_crosses_zero:
            parts.append("intervalo de confiança 95% passando pelo zero")
        observacoes = obs + "; ".join(parts) + "."
    else:
        observacoes = ""

    automl_results.append({
        "doenca": doenca,
        "modelo": model_name,
        "y_test": y_test,
        "y_pred": y_pred,
        "lower_95": lower,
        "upper_95": upper,
        "test_index": test_series_used.index,
        "summary_text": summary_text,
        "RMSE": rmse_val,
        "MAE": mae_val,
        "MAPE": mape_val,
        "observações": observacoes,
    })

print(f"Modeladas com sucesso: {len(automl_results)} séries.")

  [1/16] Abscesso da faringe
  [2/16] Deficiência enzimática da glicose-6-fosfato desidrogenase


/Users/daniellybx/Documents/Projeto ProEpi GdS /projeto_proepi_gds_datascience/venv/lib/python3.13/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency W-MON will be used.
  self._init_dates(dates, freq)
/Users/daniellybx/Documents/Projeto ProEpi GdS /projeto_proepi_gds_datascience/venv/lib/python3.13/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency W-MON will be used.
  self._init_dates(dates, freq)
/Users/daniellybx/Documents/Projeto ProEpi GdS /projeto_proepi_gds_datascience/venv/lib/python3.13/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency W-MON will be used.
  self._init_dates(dates, freq)
/Users/daniellybx/Documents/Projeto ProEpi GdS /projeto_proepi_gds_datascience/venv/lib/python3.13/site-packages/statsmodels/base/model.py:607: ConvergenceWarning:

  [3/16] Desatelação pulmonar
  [4/16] Distrofia muscular
  [5/16] Doença das células brancas
  [6/16] Doença das glândulas salivares
  [7/16] Doença do tecido conjuntivo
  [8/16] Febre do dengue
  [9/16] Hipergamalglobulinemia
  [10/16] Hipertensão arterial
  [11/16] Micotemia infecciosa
  [12/16] Poliquimia vera
  [13/16] Síndrome mielodisplásica
  [14/16] Tosse convulsa
  [15/16] Tricomonose
  [16/16] piolho
Modeladas com sucesso: 16 séries.


## 4. Save model summaries

**Summary:** Writes one Excel file per series with the chosen model’s summary (e.g. statsmodels output or model name). Path: `data/results/time_series_outputs/summaries/{doenca}_{modelo}_summary.xlsx`.

In [4]:
for r in automl_results:
    doenca = r["doenca"]
    modelo = r["modelo"]
    summary_text = r["summary_text"]
    fname = f"{safe_name(doenca)}_{safe_name(modelo)}_summary.xlsx"
    path = PATH_SUMMARIES / fname
    pd.DataFrame({"summary": [summary_text]}).to_excel(path, index=False, engine="openpyxl")
print(f"Salvos {len(automl_results)} summaries em {PATH_SUMMARIES}")

Salvos 16 summaries em /Users/daniellybx/Documents/Projeto ProEpi GdS /projeto_proepi_gds_datascience/data/results/time_series_outputs/summaries


## 5. Save charts (test series + forecast + 95% CI)

**Summary:** For each series, saves one chart showing only the **test period**: observed values, point forecast, and 95% confidence band. Path: `data/results/time_series_outputs/charts/{doenca}_{modelo}_chart.png`.

In [5]:
for r in automl_results:
    doenca = r["doenca"]
    modelo = r["modelo"]
    idx = r["test_index"]
    x_ax = idx.to_timestamp() if hasattr(idx, "to_timestamp") else idx
    y_test = r["y_test"]
    y_pred = r["y_pred"]
    lower = r["lower_95"]
    upper = r["upper_95"]
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.plot(x_ax, y_test, label="Teste (observado)", color="C0")
    ax.plot(x_ax, y_pred, label="Forecast", color="C1")
    ax.fill_between(x_ax, lower, upper, alpha=0.3, color="C1")
    ax.set_title(f"{doenca} — {modelo}")
    ax.legend(loc="best", fontsize=8)
    ax.set_ylabel("Casos")
    plt.tight_layout()
    fname = f"{safe_name(doenca)}_{safe_name(modelo)}_chart.png"
    fig.savefig(PATH_CHARTS / fname, dpi=120, bbox_inches="tight")
    plt.close(fig)
print(f"Salvos {len(automl_results)} gráficos em {PATH_CHARTS}")

Salvos 16 gráficos em /Users/daniellybx/Documents/Projeto ProEpi GdS /projeto_proepi_gds_datascience/data/results/time_series_outputs/charts


## 6. Save evaluation metrics

**Summary:** Exports a single Excel file with one row per series: disease, chosen model, RMSE, MAE, MAPE, and observations (e.g. notes when the series is poor quality). Path: `data/results/time_series_outputs/evaluation_metrics_all_models.xlsx`.

In [6]:
metrics_df = pd.DataFrame([
    {"doenca": r["doenca"], "modelo_escolhido": r["modelo"], "RMSE": r["RMSE"], "MAE": r["MAE"], "MAPE": r["MAPE"], "observações": r.get("observações", "")}
    for r in automl_results
])
path_metrics = PATH_OUTPUT / "evaluation_metrics_all_models.xlsx"
metrics_df.to_excel(path_metrics, index=False, engine="openpyxl")
print(f"Salvo: {path_metrics}")
metrics_df.head(20)

Salvo: /Users/daniellybx/Documents/Projeto ProEpi GdS /projeto_proepi_gds_datascience/data/results/time_series_outputs/evaluation_metrics_all_models.xlsx


,doenca,modelo_escolhido,RMSE,MAE,MAPE,observações
0,Abscesso da faringe,SklearnWrapper,0.0,0.0,NaN,Série muito ruim: intervalo de confiança 95% p...
1,Deficiência enzimática da glicose-6-fosfato de...,SklearnWrapper,0.0,0.0,NaN,Série muito ruim: intervalo de confiança 95% p...
2,Desatelação pulmonar,SklearnWrapper,0.0,0.0,NaN,Série muito ruim: intervalo de confiança 95% p...
3,Distrofia muscular,SklearnWrapper,0.0,0.0,NaN,Série muito ruim: intervalo de confiança 95% p...
4,Doença das células brancas,SklearnWrapper,0.0,0.0,NaN,Série muito ruim: intervalo de confiança 95% p...
5,Doença das glândulas salivares,SklearnWrapper,0.0,0.0,NaN,Série muito ruim: intervalo de confiança 95% p...
6,Doença do tecido conjuntivo,SklearnWrapper,0.0,0.0,NaN,Série muito ruim: intervalo de confiança 95% p...
7,Febre do dengue,SklearnWrapper,0.0,0.0,NaN,Série muito ruim: intervalo de confiança 95% p...
8,Hipergamalglobulinemia,SklearnWrapper,0.0,0.0,NaN,Série muito ruim: intervalo de confiança 95% p...
9,Hipertensão arterial,SklearnWrapper,0.0,0.0,NaN,Série muito ruim: intervalo de confiança 95% p...


## 7. Viable series selection summary

**Summary:** Exports an Excel file with (1) total diseases, count and list of excluded series, count and list of modeled series; (2) a per-disease table with weeks with cases and weeks filled with moving averages. Path: `data/results/time_series_outputs/evaluation_viable_series_summary.xlsx`.

In [7]:
total_diseases = len(density_df)
excluded = density_df[~density_df["doenca"].isin(viable)]["doenca"].tolist()
modeled = [r["doenca"] for r in automl_results]
n_excluded = len(excluded)
n_modeled = len(modeled)

print("=== Resumo da seleção de séries viáveis ===")
print(f"Total de doenças: {total_diseases}")
print(f"Séries excluídas: {n_excluded}")
for d in excluded:
    print(f"  - {d}")
print(f"Séries modeladas: {n_modeled}")
for d in modeled:
    print(f"  - {d}")

n_weeks = len(weeks_full)
per_disease = density_df.copy()
per_disease["n_weeks_with_cases"] = per_disease["n_weeks_with_data"]
per_disease["n_weeks_filled_ma"] = per_disease.apply(
    lambda r: (n_weeks - r["n_weeks_with_data"]) if r["doenca"] in viable else 0, axis=1
)
per_disease["total_weeks"] = n_weeks
per_disease["status"] = per_disease["doenca"].apply(
    lambda d: "modeled" if d in modeled else ("viable" if d in viable else "excluded")
)
per_disease = per_disease[["doenca", "n_weeks_with_cases", "n_weeks_filled_ma", "total_weeks", "status", "total_casos", "density"]]

path_summary = PATH_OUTPUT / "evaluation_viable_series_summary.xlsx"
with pd.ExcelWriter(path_summary, engine="openpyxl") as writer:
    summary_df = pd.DataFrame({
        "metric": ["total_diseases", "excluded_count", "modeled_count"],
        "value": [total_diseases, n_excluded, n_modeled],
    })
    summary_df.to_excel(writer, sheet_name="Summary", index=False)
    pd.DataFrame({"excluded_series": excluded}).to_excel(writer, sheet_name="Excluded_series", index=False)
    pd.DataFrame({"modeled_series": modeled}).to_excel(writer, sheet_name="Modeled_series", index=False)
    per_disease.to_excel(writer, sheet_name="Per_disease", index=False)
print(f"Salvo: {path_summary}")
per_disease.head(15)

=== Resumo da seleção de séries viáveis ===
Total de doenças: 86
Séries excluídas: 70
  - Abscesso peritonsiliano
  - Acariose
  - Anemia aplásica
  - Anemia hemolítica
  - Aneurisma ao torácico da aorta
  - Cefaleia de tensão
  - Celulite orbital
  - Cistite
  - Colangite ascendente
  - Colite ulcerativa
  - Conjuntivite bacteriana
  - Câncer intestinal
  - Câncer páncreático
  - Câncer tireoideano
  - Deficiência de glicocorticoide
  - Deficiência de ácido fólico
  - Desabsorção Intestinal
  - Doença Metabólica
  - Doença de Lyme
  - Doença do Joelho Temporomandibular
  - Doença nasal
  - Doença parasítica
  - Esclerite
  - Esclerodermia
  - Faringite
  - Febre tifóide
  - Fevre rubra
  - Fibrose cística
  - Fibrose pulmonar
  - Gastrite
  - Gastroenterite infectosa
  - Gripe
  - Hematomia
  - Hemocromatose
  - Hemofilia
  - Hepatite gordurosa aguda do período de gravidez (HLAP)
  - Hepatite induzida por toxina
  - Hipernatremia
  - Hipertensão Pulmonar
  - Hipotireoidismo
  - Impeti

,doenca,n_weeks_with_cases,n_weeks_filled_ma,total_weeks,status,total_casos,density
0,Abscesso da faringe,32,20,52,modeled,294,0.615385
1,Abscesso peritonsiliano,4,0,52,excluded,4,0.076923
2,Acariose,10,0,52,excluded,12,0.192308
3,Anemia aplásica,9,0,52,excluded,13,0.173077
4,Anemia hemolítica,28,0,52,excluded,194,0.538462
5,Aneurisma ao torácico da aorta,8,0,52,excluded,9,0.153846
6,Cefaleia de tensão,1,0,52,excluded,1,0.019231
7,Celulite orbital,1,0,52,excluded,1,0.019231
8,Cistite,12,0,52,excluded,18,0.230769
9,Colangite ascendente,1,0,52,excluded,1,0.019231
